# SureLink Gold Dimensional Model


In [3]:
from pyspark.sql import functions as F

claim = spark.read.table("Claim")
policy = spark.read.table("Policy")
customer = spark.read.table("Customer")
vehicle = spark.read.table("Vehicle")

print("Claim rows:", claim.count())
print("Policy rows:", policy.count())
print("Customer rows:", customer.count())
print("Vehicle rows:", vehicle.count())

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 5, Finished, Available, Finished, False)

Claim rows: 19931
Policy rows: 20597
Customer rows: 17171
Vehicle rows: 20597


### Check validity of relationships

In [4]:
# Check claims without a matching policy

unmatched_claims = (
    claim.alias("c")
    .join(
        policy.select("policy_id").alias("p"),
        F.col("c.policy_id") == F.col("p.policy_id"),
        "left"
    )
    .filter(F.col("p.policy_id").isNull())
)

print(
    "Claims without a matching policy:",
    unmatched_claims.count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 6, Finished, Available, Finished, False)

Claims without a matching policy: 0


In [5]:
unmatched_customers = (
    policy.alias("p")
    .join(
        customer.select("customer_id").alias("c"),
        F.col("p.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .filter(F.col("c.customer_id").isNull())
)

print(
    "Policies without a matching customer:",
    unmatched_customers.count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 7, Finished, Available, Finished, False)

Policies without a matching customer: 0


In [6]:
unmatched_vehicles = (
    policy.alias("p")
    .join(
        vehicle.select("vehicle_id").alias("v"),
        F.col("p.vehicle_id") == F.col("v.vehicle_id"),
        "left"
    )
    .filter(F.col("v.vehicle_id").isNull())
)

print(
    "Policies without a matching vehicle:",
    unmatched_vehicles.count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 8, Finished, Available, Finished, False)

Policies without a matching vehicle: 0


### Check duplicate IDs

In [7]:
def check_duplicate_keys(df, key):
    return (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

print("Duplicate claim IDs:", check_duplicate_keys(claim, "claim_id"))
print("Duplicate policy IDs:", check_duplicate_keys(policy, "policy_id"))
print("Duplicate customer IDs:", check_duplicate_keys(customer, "customer_id"))
print("Duplicate vehicle IDs:", check_duplicate_keys(vehicle, "vehicle_id"))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 9, Finished, Available, Finished, False)

Duplicate claim IDs: 0
Duplicate policy IDs: 0
Duplicate customer IDs: 0
Duplicate vehicle IDs: 0


### **Create FactClaim**

In [8]:
from pyspark.sql import functions as F

claim = spark.read.table("Claim")
policy = spark.read.table("Policy")

fact_claim = (
    claim.alias("c")
    .join(
        policy.alias("p"),
        F.col("c.policy_id") == F.col("p.policy_id"),
        "left"
    )
    .select(
        F.col("c.claim_id"),
        F.col("c.policy_id"),
        F.col("p.customer_id"),
        F.col("p.vehicle_id"),
        F.col("c.incident_date"),
        F.col("c.total_claim_amount"),
        F.col("c.approved_amount"),
        F.col("c.claim_status"),
        F.col("c.incident_type"),
        F.col("c.incident_severity"),
        F.col("c.incident_province"),
        F.col("p.cover_level"),
        F.col("p.annual_premium")
    )
)

print("FactClaim rows:", fact_claim.count())

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 10, Finished, Available, Finished, False)

FactClaim rows: 19931


### **Save FactClaim**

In [9]:
# Register the table in the Gold Lakehouse, not the default Silver Lakehouse.
target_table = "SureLink.SureLink_Gold_lh.dbo.FactClaim"

(
    fact_claim.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Registered Gold table:", target_table)


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 11, Finished, Available, Finished, False)

Registered Gold table: SureLink.SureLink_Gold_lh.dbo.FactClaim


### Verify the saved table

In [10]:
saved_fact_claim = spark.table("SureLink.SureLink_Gold_lh.dbo.FactClaim")

print("Registered FactClaim rows:", saved_fact_claim.count())
display(saved_fact_claim.limit(10))


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 12, Finished, Available, Finished, False)

Registered FactClaim rows: 19931


SynapseWidget(Synapse.DataFrame, 35a8c928-63dc-477c-bab2-52a6052d9c5c)

### **Build DimCustomer Dimensional Table**

## Load the Silver Customer table

In [11]:
customer = spark.read.table("Customer")

print("Customer rows:", customer.count())

display(customer.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 13, Finished, Available, Finished, False)

Customer rows: 17171


SynapseWidget(Synapse.DataFrame, dc8a44d4-b85f-4e0b-b7fb-94c7696178b4)

In [12]:
dim_customer = customer.select(
    "customer_id",
    "customer_ref",
    "first_name",
    "last_name",
    "gender",
    "birth_date",
    "occupation",
    "education_level",
    "city",
    "province",
    "postal_code"
)

print("DimCustomer rows:", dim_customer.count())

display(dim_customer.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 14, Finished, Available, Finished, False)

DimCustomer rows: 17171


SynapseWidget(Synapse.DataFrame, eb5670b5-5b0d-4866-bc1e-e5b4011a809e)

### Validate DimCustomer

In [13]:
## check for duplicate & missing customer IDs

print(
    "Duplicate customer IDs:",
    dim_customer.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing customer IDs:",
    dim_customer.filter(
        F.col("customer_id").isNull()
    ).count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 15, Finished, Available, Finished, False)

Duplicate customer IDs: 0
Missing customer IDs: 0


### Save DimCustomer to SureLink_Gold

In [14]:
# Register the table in the Gold Lakehouse, not the default Silver Lakehouse.
target_table = "SureLink.SureLink_Gold_lh.dbo.DimCustomer"

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Registered Gold table:", target_table)


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 16, Finished, Available, Finished, False)

Registered Gold table: SureLink.SureLink_Gold_lh.dbo.DimCustomer


### Verify the saved table

In [15]:
saved_dim_customer = spark.table("SureLink.SureLink_Gold_lh.dbo.DimCustomer")

print("Registered DimCustomer rows:", saved_dim_customer.count())
display(saved_dim_customer.limit(10))


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 17, Finished, Available, Finished, False)

Registered DimCustomer rows: 17171


SynapseWidget(Synapse.DataFrame, 4eff842a-23a4-4129-a379-0b70bc665386)

## **Build DimVehicle Build DimVehicle**

### Load the Silver Vehicle table

In [16]:
vehicle = spark.read.table("Vehicle")

print("Vehicle rows:", vehicle.count())

display(vehicle.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 18, Finished, Available, Finished, False)

Vehicle rows: 20597


SynapseWidget(Synapse.DataFrame, bf7097c8-7390-4ad4-84e9-75f69b6af42c)

In [17]:
dim_vehicle = vehicle.select(
    "vehicle_id",
    "make",
    "model",
    "model_year"
)

print("DimVehicle rows:", dim_vehicle.count())

display(dim_vehicle.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 19, Finished, Available, Finished, False)

DimVehicle rows: 20597


SynapseWidget(Synapse.DataFrame, 7f62b6ac-e962-4035-949c-be9d734f6f03)

### Validate the dimension

In [18]:
print(
    "Duplicate vehicle IDs:",
    dim_vehicle.groupBy("vehicle_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing vehicle IDs:",
    dim_vehicle.filter(
        F.col("vehicle_id").isNull()
    ).count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 20, Finished, Available, Finished, False)

Duplicate vehicle IDs: 0
Missing vehicle IDs: 0


### Save DimVehicle to SureLink_Gold

In [19]:
# Register the table in the Gold Lakehouse, not the default Silver Lakehouse.
target_table = "SureLink.SureLink_Gold_lh.dbo.DimVehicle"

(
    dim_vehicle.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Registered Gold table:", target_table)


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 21, Finished, Available, Finished, False)

Registered Gold table: SureLink.SureLink_Gold_lh.dbo.DimVehicle


### Verify the saved table

In [20]:
saved_dim_vehicle = spark.table("SureLink.SureLink_Gold_lh.dbo.DimVehicle")

print("Registered DimVehicle rows:", saved_dim_vehicle.count())
display(saved_dim_vehicle.limit(10))


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 22, Finished, Available, Finished, False)

Registered DimVehicle rows: 20597


SynapseWidget(Synapse.DataFrame, 1bb926cc-1d4d-45ba-a6ce-7b248107fc0a)

### **Build DimPolicy**

### Load Silver Policy

In [21]:
policy = spark.read.table("Policy")

print("Policy rows:", policy.count())

display(policy.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 23, Finished, Available, Finished, False)

Policy rows: 20597


SynapseWidget(Synapse.DataFrame, 606caba7-94d5-43cf-93b6-5f9f5a916e19)

In [22]:
dim_policy = policy.select(
    "policy_id",
    "policy_number",
    "customer_id",
    "vehicle_id",
    "bind_date",
    "cover_level",
    "excess_amount",
    "annual_premium",
    "extended_liability_limit",
    "policy_status"
)

print("DimPolicy rows:", dim_policy.count())

display(dim_policy.limit(5))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 24, Finished, Available, Finished, False)

DimPolicy rows: 20597


SynapseWidget(Synapse.DataFrame, 73c77463-64ff-4e54-bc44-e8ebefb99b03)

### Validate the policy key

In [23]:
print(
    "Duplicate policy IDs:",
    dim_policy.groupBy("policy_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing policy IDs:",
    dim_policy.filter(
        F.col("policy_id").isNull()
    ).count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 25, Finished, Available, Finished, False)

Duplicate policy IDs: 0
Missing policy IDs: 0


### Save DimPolicy

In [24]:
# Register the table in the Gold Lakehouse, not the default Silver Lakehouse.
target_table = "SureLink.SureLink_Gold_lh.dbo.DimPolicy"

(
    dim_policy.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Registered Gold table:", target_table)


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 26, Finished, Available, Finished, False)

Registered Gold table: SureLink.SureLink_Gold_lh.dbo.DimPolicy


### Verify the saved table

In [25]:
saved_dim_policy = spark.table("SureLink.SureLink_Gold_lh.dbo.DimPolicy")

print("Registered DimPolicy rows:", saved_dim_policy.count())
display(saved_dim_policy.limit(10))


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 27, Finished, Available, Finished, False)

Registered DimPolicy rows: 20597


SynapseWidget(Synapse.DataFrame, 4dd4f3a8-ff7e-4ce1-8f85-aade2addb55c)

## **Build DimDate**

In [26]:
## Check the date range

date_range = fact_claim.select(
    F.min("incident_date").alias("earliest_date"),
    F.max("incident_date").alias("latest_date")
)

display(date_range)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38ca0168-4742-4cf2-87dc-01892bc0b55e)

In [27]:
## Generate DimDate

from pyspark.sql import functions as F

date_bounds = fact_claim.select(
    F.min("incident_date").alias("start_date"),
    F.max("incident_date").alias("end_date")
).first()

dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_bounds["start_date"]),
                F.lit(date_bounds["end_date"]),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
    .select(
        F.col("date").alias("date_key"),
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month_number"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.dayofmonth("date").alias("day_of_month"),
        F.dayofweek("date").alias("day_of_week"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.date_format("date", "yyyy-MM").alias("year_month")
    )
)

print("DimDate rows:", dim_date.count())

display(dim_date.limit(10))

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 29, Finished, Available, Finished, False)

DimDate rows: 389


SynapseWidget(Synapse.DataFrame, 904f3b39-bb84-4c86-afba-b9d8d18b3ce9)

### Validate DimDate

In [28]:
print(
    "Duplicate dates:",
    dim_date.groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing dates:",
    dim_date.filter(
        F.col("date_key").isNull()
    ).count()
)

print(
    "Total rows:",
    dim_date.count()
)

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 30, Finished, Available, Finished, False)

Duplicate dates: 0
Missing dates: 0
Total rows: 389


### Save DimDate to SureLink_Gold

In [29]:
# Register the table in the Gold Lakehouse, not the default Silver Lakehouse.
target_table = "SureLink.SureLink_Gold_lh.dbo.DimDate"

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Registered Gold table:", target_table)


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 31, Finished, Available, Finished, False)

Registered Gold table: SureLink.SureLink_Gold_lh.dbo.DimDate


### Verify the saved table

In [30]:
saved_dim_date = spark.table("SureLink.SureLink_Gold_lh.dbo.DimDate")

print("Registered DimDate rows:", saved_dim_date.count())
display(saved_dim_date.limit(10))


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 32, Finished, Available, Finished, False)

Registered DimDate rows: 389


SynapseWidget(Synapse.DataFrame, 252aa5db-ee63-42d7-a281-d8647d944d58)

## **Build relationships between the tables**

### Load the saved Gold tables

In [31]:
# Read the registered tables from SureLink_Gold_lh.
gold_fact_claim = spark.table("SureLink.SureLink_Gold_lh.dbo.FactClaim")
gold_dim_customer = spark.table("SureLink.SureLink_Gold_lh.dbo.DimCustomer")
gold_dim_vehicle = spark.table("SureLink.SureLink_Gold_lh.dbo.DimVehicle")
gold_dim_policy = spark.table("SureLink.SureLink_Gold_lh.dbo.DimPolicy")
gold_dim_date = spark.table("SureLink.SureLink_Gold_lh.dbo.DimDate")

print("All five registered Gold tables loaded successfully.")


StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 33, Finished, Available, Finished, False)

All five registered Gold tables loaded successfully.


### Check the date relationship

In [32]:
unmatched_dates = (
    gold_fact_claim.alias("f")
    .join(
        gold_dim_date.select("date_key").alias("d"),
        F.col("f.incident_date") == F.col("d.date_key"),
        "left"
    )
    .filter(F.col("d.date_key").isNull())
)

print("Claims without a matching date:", unmatched_dates.count())



StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 34, Finished, Available, Finished, False)

Claims without a matching date: 0


### validate all dimension relationships

In [33]:
checks = [
    ("Customer", gold_fact_claim, "customer_id", gold_dim_customer, "customer_id"),
    ("Vehicle", gold_fact_claim, "vehicle_id", gold_dim_vehicle, "vehicle_id"),
    ("Policy", gold_fact_claim, "policy_id", gold_dim_policy, "policy_id"),
]

for name, fact_df, fact_key, dim_df, dim_key in checks:
    unmatched = (
        fact_df.alias("f")
        .join(
            dim_df.select(dim_key).alias("d"),
            F.col(f"f.{fact_key}") == F.col(f"d.{dim_key}"),
            "left"
        )
        .filter(F.col(f"d.{dim_key}").isNull())
        .count()
    )

    print(f"Claims without a matching {name}: {unmatched}")

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 35, Finished, Available, Finished, False)

Claims without a matching Customer: 0
Claims without a matching Vehicle: 0
Claims without a matching Policy: 0


### Verify the complete Gold layer

In [34]:
gold_tables = {
    "FactClaim": gold_fact_claim,
    "DimCustomer": gold_dim_customer,
    "DimVehicle": gold_dim_vehicle,
    "DimPolicy": gold_dim_policy,
    "DimDate": gold_dim_date
}

for table_name, df in gold_tables.items():
    print(f"{table_name}: {df.count():,} rows")

print("Duplicate claim IDs:", 
      gold_fact_claim.count() - 
      gold_fact_claim.select("claim_id").distinct().count())

print("Missing claim IDs:", 
      gold_fact_claim.filter(
          F.col("claim_id").isNull()
      ).count())

StatementMeta(, 7abf0104-5447-4e33-9326-7c763562787e, 36, Finished, Available, Finished, False)

FactClaim: 19,931 rows
DimCustomer: 17,171 rows
DimVehicle: 20,597 rows
DimPolicy: 20,597 rows
DimDate: 389 rows
Duplicate claim IDs: 0
Missing claim IDs: 0
